[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/BMGLab/BFB/blob/main/W04_Sequence_similarity_without_overclaiming.ipynb)

# Week 04 | Sequence similarity without overclaiming

**Core practical: 45 minutes.** Distinguish an alignment score from evidence of shared function.

No paid AI tool, local installation or external dataset download is required. Open it in Colab with the badge above, then **File > Save a copy in Drive** before you start so your work is kept. Run the cells from top to bottom. Local Jupyter with Python 3.9+ is an alternative. Plotting is optional.

**Data boundary:** Every generated value is synthetic. Explicit REAL EXCERPT / REAL record cards are separately labelled and limited to their stated purpose. No patient data should be entered.

## Before running (5 min)
DNA sequences and protein sequences use different alphabets; compare like with like.

Write a prediction in the response cell before executing the analysis.

## Setup (5 min)
Replace only `COURSE_ID` with your assigned pseudonym. A seed supports reproducibility; it is not proof of authorship.

In [ ]:
import hashlib, json, math, random, statistics, sys
from pathlib import Path
COURSE_ID = "demo-001"  # Replace with your assigned course pseudonym, not your name or national ID.
SEED = int(hashlib.sha256(COURSE_ID.encode()).hexdigest()[:8], 16)
rng = random.Random(SEED)
RESULTS = {}
print("Python", sys.version.split()[0], "| course ID", COURSE_ID, "| seed", SEED)

def mean(values):
    if not values: raise ValueError("Cannot average an empty list")
    return sum(values) / len(values)

def bh_adjust(pvalues):
    """Benjamini-Hochberg adjusted p-values, returned in original order."""
    if any(not 0 <= p <= 1 for p in pvalues): raise ValueError("p must be in [0,1]")
    m = len(pvalues)
    order = sorted(range(m), key=lambda i: pvalues[i])
    out = [0.0] * m
    running = 1.0
    for j in range(m - 1, -1, -1):
        i = order[j]
        running = min(running, pvalues[i] * m / (j + 1))
        out[i] = running
    return out

def optional_plot(labels, values, ylabel, title):
    """Plot when matplotlib is available; numerical work never requires it."""
    try:
        import matplotlib.pyplot as plt
    except ImportError:
        print("Plot unavailable; use the numerical table above.")
        return
    fig, ax = plt.subplots(figsize=(7, 4))
    ax.bar(labels, values)
    ax.set(ylabel=ylabel, title=title)
    fig.tight_layout()
    plt.show()

### Who is submitting (1 min)
Fill in your **full name** and your **Ege student number**, then run the cell. It refuses to
continue if either is missing or malformed, so a mistyped digit is caught here — in the room,
where it takes ten seconds to fix — rather than after the deadline.

Your `COURSE_ID` above still deals your dataset. This is only about attributing the work to you.

In [ ]:
import os
if not os.path.exists("bib_colab.py"):
    !curl -sfO https://raw.githubusercontent.com/BMGLab/BFB/main/bib_colab.py
import bib_colab as bib

A = bib.start("W04")
A.whoami(
    name="",          # your full name, e.g. "Ayşe Gül Öztürk"
    student_no="",    # your Ege student number, digits only
    section="EN",     # "EN" or "TR"
)

### Prediction
Write your prediction here before running the investigation, then copy it into PREDICTION in the response cell.

## Guided investigation (20 min)
1. Score a fixed alignment using +2 match, -1 mismatch.
2. Inspect a small exact local-alignment score matrix.
3. Change the mismatch penalty, not the biology.
4. Compare the score with a short evidence interpretation.

In [ ]:
def fixed_score(a, b, match=2, mismatch=-1):
    if len(a) != len(b): raise ValueError("This function only scores equal-length ungapped strings")
    return sum(match if x == y else mismatch for x, y in zip(a, b))

def local_score(a, b, match=2, mismatch=-1, gap=-2):
    # Smith-Waterman with a linear gap score; score only, no traceback.
    H = [[0] * (len(b)+1) for _ in range(len(a)+1)]
    for i in range(1, len(a)+1):
        for j in range(1, len(b)+1):
            H[i][j] = max(0, H[i-1][j-1] + (match if a[i-1] == b[j-1] else mismatch),
                           H[i-1][j] + gap, H[i][j-1] + gap)
    return max(map(max, H)), H

a, b = "ACGT", "ACCT"
score, matrix = local_score(a, b)
for row in matrix: print(row)
identity = sum(x == y for x,y in zip(a,b)) / len(a)
print("Fixed score", fixed_score(a,b), "fixed identity", identity, "best local score", score)
print("Fixed score with mismatch -4:", fixed_score(a,b,mismatch=-4))
assert fixed_score(a,b) == 5
assert local_score("AC", "AC")[0] == 4
assert local_score("A", "T")[0] == 0
RESULTS = {"data_status": "SYNTHETIC", "fixed_score": fixed_score(a,b), "fixed_identity": identity,
           "best_local_score": score}

## Explain the evidence (10 min)
**Q1.** Show how the fixed alignment score of ACGT versus ACCT is calculated.

**Q2.** Why is 75% identity not a probability that two proteins have the same function?

**Q3.** What information would you report alongside a real BLAST hit before making a functional claim?

In [ ]:
PREDICTION = ""  # Fill before the analysis.
RESPONSES = {"Q1": "", "Q2": "", "Q3": ""}
AI_DISCLOSURE = "No AI used."  # Change to tool, date, purpose, and checks if you used one.
CHECK_PERFORMED = ""  # Describe one actual check, even if it found no error.

## Save and submit (5 min)
Complete your responses above, then **Runtime > Restart session and run all** so every number in
the notebook is the one your answers describe.

Then run the cell below. It checks that nothing is missing, prints a receipt code, and sends this
notebook straight to your instructor. There is nothing to download and nothing to upload.

A completion check looks for the presence of your responses, not for scientific correctness. If
the upload fails, the cell prints your receipt code and what to do instead — follow it before you
leave. Paper fallback may be submitted as a legible scan with the same answers.

In [ ]:
required = [PREDICTION, CHECK_PERFORMED] + list(RESPONSES.values())
complete = all(isinstance(x,str) and x.strip() for x in required)
safe_id = "".join(c for c in COURSE_ID if c.isalnum() or c in "-_")[:40] or "anonymous"
report = {"week":4, "course_id":COURSE_ID,"seed":SEED,"python":sys.version.split()[0],
          "prediction":PREDICTION,"results":RESULTS,"responses":RESPONSES,
          "check_performed":CHECK_PERFORMED,"AI_disclosure":AI_DISCLOSURE,
          "response_fields_complete":complete}
output = Path(f"W04_{safe_id}_summary.json")
output.write_text(json.dumps(report,indent=2),encoding="utf-8")
print("Saved:",output)
print("Ready for review" if complete else "DRAFT: fill prediction, responses and check before submission")

# --- submit -------------------------------------------------------------
A.answers(RESPONSES,
          prediction=PREDICTION,
          check=CHECK_PERFORMED,
          disclosure=AI_DISCLOSURE,
          results=RESULTS)
A.check()
A.submit()

## Paper / device-free route
Write the two four-base sequences, mark three matches, calculate 5. Compare to a hypothetical 75%-identity match covering only four bases of a long protein.

## Optional extension
Optional: inspect affine gap penalties and substitution matrices; deriving log-odds is not examined.

## Sources
- [S06] NCBI: How BLAST Works and BLAST Statistics: The Expect Value. https://www.nlm.nih.gov/ncbi/workshops/2023-08_BLAST_evol/e_value.html
- [S26] NCBI: How BLAST Works (2022 workshop). https://www.nlm.nih.gov/ncbi/workshops/2022-10_Basic-Web-BLAST/how-blast-works.html